# 🔍 Semantic Search

**Search by meaning, not just keywords**

---

## 📋 Overview

**What you'll learn:**
- Semantic vs keyword search
- Building a semantic search engine
- Similarity metrics
- Ranking and reranking
- Production semantic search

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict, Tuple
import chromadb

print("✅ Setup complete")

## 🆚 Keyword vs Semantic Search

### Keyword Search (Traditional):
```python
Query: "Python programming"
Matches: Documents containing "Python" AND "programming"

❌ Misses: "Coding in Python", "Python development"
```

### Semantic Search (Vector-based):
```python
Query: "Python programming"
Matches: Documents about programming in Python, regardless of exact words

✅ Finds: "Coding in Python", "Python development", "Write Python code"
```

### Why Semantic Search?
- 🎯 **Meaning over keywords**: Find related content
- 🌍 **Language agnostic**: Works across languages
- 🔄 **Synonyms**: Automatically handles variations
- 📊 **Better recall**: Don't miss relevant results

## 🏗️ Building a Semantic Search Engine

In [ ]:
class SemanticSearch:
    """Simple semantic search engine."""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Initialize with embedding model."""
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
        
        print(f"✅ Loaded model: {model_name}")
    
    def index(self, documents: List[str]):
        """Index documents by creating embeddings."""
        self.documents = documents
        
        print(f"Creating embeddings for {len(documents)} documents...")
        self.embeddings = self.model.encode(documents, show_progress_bar=True)
        
        print(f"✅ Indexed {len(documents)} documents")
        print(f"   Embedding shape: {self.embeddings.shape}")
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for most similar documents."""
        if self.embeddings is None:
            raise ValueError("No documents indexed. Call index() first.")
        
        # Encode query
        query_embedding = self.model.encode([query])[0]
        
        # Calculate cosine similarity
        similarities = self._cosine_similarity(query_embedding, self.embeddings)
        
        # Get top k
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'document': self.documents[idx],
                'score': float(similarities[idx]),
                'rank': len(results) + 1
            })
        
        return results
    
    def _cosine_similarity(self, query_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
        """Calculate cosine similarity between query and all documents."""
        # Normalize vectors
        query_norm = query_vec / np.linalg.norm(query_vec)
        doc_norms = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
        
        # Dot product = cosine similarity (for normalized vectors)
        similarities = np.dot(doc_norms, query_norm)
        
        return similarities

# Create search engine
search_engine = SemanticSearch()

# Sample documents
documents = [
    "Python is a high-level programming language",
    "Machine learning models can predict future outcomes",
    "Natural language processing helps computers understand text",
    "Deep learning uses neural networks with multiple layers",
    "Data science combines statistics and programming",
    "Artificial intelligence mimics human intelligence",
    "Cloud computing provides on-demand computing resources",
    "JavaScript is used for web development",
    "SQL is a language for database queries",
    "APIs allow different software systems to communicate",
]

# Index documents
search_engine.index(documents)

In [ ]:
# Test semantic search
queries = [
    "AI and neural networks",
    "coding in Python",
    "understanding human language"
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    print("="*60)
    
    results = search_engine.search(query, top_k=3)
    
    for result in results:
        print(f"  #{result['rank']} (score: {result['score']:.3f})")
        print(f"     {result['document']}")

## 📊 Similarity Metrics

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity: measures angle between vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Euclidean distance: straight-line distance."""
    return np.linalg.norm(a - b)

def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Dot product: unnormalized similarity."""
    return np.dot(a, b)

# Compare metrics
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [
    "I love programming",
    "I enjoy coding",           # Very similar
    "The weather is nice",      # Completely different
]

embeddings = model.encode(texts)

print("📊 Comparing Similarity Metrics\n")
print(f"Text 1: {texts[0]}")
print(f"Text 2: {texts[1]}")
print(f"Text 3: {texts[2]}\n")

print("Similarity of Text 1 vs Text 2 (similar):")
print(f"  Cosine:    {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"  Euclidean: {euclidean_distance(embeddings[0], embeddings[1]):.4f}")
print(f"  Dot:       {dot_product(embeddings[0], embeddings[1]):.4f}")

print("\nSimilarity of Text 1 vs Text 3 (different):")
print(f"  Cosine:    {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"  Euclidean: {euclidean_distance(embeddings[0], embeddings[2]):.4f}")
print(f"  Dot:       {dot_product(embeddings[0], embeddings[2]):.4f}")

print("\n💡 Cosine similarity is most commonly used (range: -1 to 1)")

## 🎯 Advanced: Reranking

In [ ]:
class RerankedSemanticSearch(SemanticSearch):
    """Semantic search with reranking."""
    
    def search_with_rerank(
        self,
        query: str,
        top_k: int = 5,
        rerank_top_n: int = 20
    ) -> List[Dict]:
        """
        Two-stage retrieval:
        1. Fast: Get top N candidates
        2. Accurate: Rerank top K
        """
        
        # Stage 1: Fast retrieval
        candidates = self.search(query, top_k=rerank_top_n)
        
        # Stage 2: Rerank using cross-encoder
        # For demo, we'll use a simple scoring boost
        # In production, use a cross-encoder model
        
        for candidate in candidates:
            # Boost score if query words appear in document
            query_words = set(query.lower().split())
            doc_words = set(candidate['document'].lower().split())
            
            overlap = len(query_words & doc_words)
            candidate['rerank_score'] = candidate['score'] + (overlap * 0.1)
        
        # Sort by reranked score
        candidates.sort(key=lambda x: x['rerank_score'], reverse=True)
        
        # Return top k
        return candidates[:top_k]

# Test reranking
reranked_engine = RerankedSemanticSearch()
reranked_engine.index(documents)

query = "programming language"

print(f"🔍 Query: '{query}'\n")

print("Standard semantic search:")
standard_results = reranked_engine.search(query, top_k=3)
for r in standard_results:
    print(f"  {r['rank']}. ({r['score']:.3f}) {r['document'][:50]}...")

print("\nWith reranking:")
reranked_results = reranked_engine.search_with_rerank(query, top_k=3)
for i, r in enumerate(reranked_results, 1):
    print(f"  {i}. ({r['rerank_score']:.3f}) {r['document'][:50]}...")

## 🚀 Production: ChromaDB Integration

In [ ]:
class ProductionSemanticSearch:
    """Production semantic search with ChromaDB."""
    
    def __init__(self, collection_name: str = "semantic_search"):
        self.client = chromadb.Client()
        
        # Create or get collection
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        print(f"✅ Connected to collection: {collection_name}")
    
    def index(
        self,
        documents: List[str],
        metadatas: List[Dict] = None,
        ids: List[str] = None
    ):
        """Index documents with metadata."""
        
        if ids is None:
            ids = [f"doc_{i}" for i in range(len(documents))]
        
        if metadatas is None:
            metadatas = [{"index": i} for i in range(len(documents))]
        
        self.collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        
        print(f"✅ Indexed {len(documents)} documents")
    
    def search(
        self,
        query: str,
        top_k: int = 5,
        where: Dict = None
    ) -> List[Dict]:
        """Search with optional metadata filtering."""
        
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k,
            where=where
        )
        
        # Format results
        formatted = []
        for i in range(len(results['ids'][0])):
            formatted.append({
                'id': results['ids'][0][i],
                'document': results['documents'][0][i],
                'score': 1 - results['distances'][0][i],  # Convert distance to similarity
                'metadata': results['metadatas'][0][i],
                'rank': i + 1
            })
        
        return formatted
    
    def clear(self):
        """Clear all documents."""
        self.client.delete_collection(self.collection.name)
        print(f"✅ Cleared collection")

# Create production search engine
prod_search = ProductionSemanticSearch(collection_name="tech_docs")

# Index with metadata
documents_with_meta = [
    {"text": "Python is great for data science", "category": "programming", "year": 2024},
    {"text": "Machine learning predicts patterns", "category": "ai", "year": 2024},
    {"text": "JavaScript powers web applications", "category": "programming", "year": 2023},
    {"text": "Deep learning uses neural networks", "category": "ai", "year": 2024},
    {"text": "Cloud services are scalable", "category": "infrastructure", "year": 2023},
]

texts = [doc["text"] for doc in documents_with_meta]
metadatas = [{"category": doc["category"], "year": doc["year"]} for doc in documents_with_meta]

prod_search.index(texts, metadatas=metadatas)

# Search with filter
print("\n🔍 Search: 'neural networks' in AI category")
results = prod_search.search(
    "neural networks",
    top_k=3,
    where={"category": "ai"}
)

for r in results:
    print(f"  {r['rank']}. ({r['score']:.3f}) {r['document']}")
    print(f"      {r['metadata']}")

## 📈 Performance Comparison

In [ ]:
import time

def benchmark_search(engine, queries: List[str], top_k: int = 5) -> Dict:
    """Benchmark search performance."""
    
    start = time.time()
    
    for query in queries:
        results = engine.search(query, top_k=top_k)
    
    elapsed = time.time() - start
    avg_latency = elapsed / len(queries)
    
    return {
        'total_time': elapsed,
        'avg_latency': avg_latency,
        'queries_per_sec': len(queries) / elapsed
    }

# Benchmark queries
benchmark_queries = [
    "programming languages",
    "artificial intelligence",
    "web development",
    "data analysis",
    "cloud computing"
]

print("⚡ Benchmarking Search Performance\n")

stats = benchmark_search(search_engine, benchmark_queries * 10)

print(f"Queries: {len(benchmark_queries) * 10}")
print(f"Total time: {stats['total_time']:.3f}s")
print(f"Avg latency: {stats['avg_latency']*1000:.1f}ms")
print(f"Throughput: {stats['queries_per_sec']:.1f} queries/sec")

## ✅ Summary

### Key Concepts:

1. **🔍 Semantic Search**
   - Search by meaning, not keywords
   - Uses vector embeddings
   - Better recall and relevance

2. **📊 Similarity Metrics**
   - **Cosine**: Most common (angle between vectors)
   - **Euclidean**: Distance in space
   - **Dot product**: Unnormalized similarity

3. **🎯 Reranking**
   - Two-stage retrieval
   - Fast + Accurate
   - Better precision

4. **🚀 Production**
   - Use vector databases (ChromaDB, Pinecone)
   - Add metadata filtering
   - Monitor performance

### Architecture:

```
Documents → Embeddings → Vector DB
                              ↓
Query → Embedding → Search → Results
```

### When to Use Semantic Search:

✅ **Use when:**
- Need to find similar content
- Synonyms and paraphrases matter
- Cross-language search
- Question answering

❌ **Don't use when:**
- Exact match required (use keyword search)
- Very small dataset (< 100 docs)
- Real-time indexing needed

### Performance Tips:

1. **Index Once**: Embeddings are expensive
2. **Batch Queries**: Process multiple at once
3. **Use ANN**: Approximate Nearest Neighbors (HNSW)
4. **Cache Results**: For common queries
5. **Right Model**: Balance speed vs accuracy

### Model Selection:

| Model | Dimensions | Speed | Accuracy | Use Case |
|-------|------------|-------|----------|----------|
| MiniLM | 384 | ⚡⚡⚡ | Good | General |
| MPNet | 768 | ⚡⚡ | Better | Quality matters |
| E5-large | 1024 | ⚡ | Best | Research |

### Production Checklist:

- ✅ Use vector database (ChromaDB, Pinecone, Weaviate)
- ✅ Add metadata for filtering
- ✅ Implement caching
- ✅ Monitor query latency
- ✅ Track relevance metrics
- ✅ A/B test different models

### Next: `04_embeddings_vectors/05_hybrid_search.ipynb`